In [ ]:
# Importar librerias
print('Leyendo librerias')
import urllib.parse
import os
import pandas as pd
import numpy as np
import polars as pl
import fastexcel
import pyarrow as pa
import pyarrow.parquet as pq
import time
print("Librerias leidas") 

In [ ]:
# Leyendo tablas
ventas_cashea = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\cashea_bills.parquet')
tasas = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\cashea_transactions_coin.parquet')
productos = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\products.parquet')
usuarios = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\users.parquet')
financiamientos = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\finance_cashea.parquet')
pagos = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\pays_cashea.parquet')
telefonos_vendidos_cashea = pl.read_parquet(r'C:\Users\POWERBI\Documents\Stored Data Parquet\phones_cashea.parquet')
envios_mrw = pl.read_excel(r'C:\Users\POWERBI\Documents\Cashea\2026.xlsx', sheet_name='Data')

In [ ]:
# Trabajando con los join de las datas
################################################################################################################
# Trabajando con los join de la data para las facturaciones
################################################################################################################
bill_report = ventas_cashea.join(tasas, on='line', how='inner')
bill_report = bill_report.drop(['line', 'fecha', 'hora'])
bill_report = bill_report.join(productos, left_on='code_product', right_on='code', how='inner')
bill_report = bill_report.drop(['department', 'mark', 'status'])
bill_report = bill_report.join(usuarios, left_on='user_code', right_on='code', how='inner')
bill_report = bill_report.with_columns(
    (pl.col('total_net') / pl.col('tasa_autocompletada')).round(2)
    .alias('dollar_net')
)
bill_report = bill_report.with_columns(
    pl.col('register_hour').dt.hour()
    .alias('hora')
)
bill_report = bill_report.drop(['user_code', 'code_product', 'total_net', 'tasa_autocompletada'])
bill_report_backup = bill_report
################################################################################################################
# Trabajando con los join de la data para los financiamientos
################################################################################################################
financing_report = financiamientos.join(pagos, on='factura', how='left')
financing_report = financing_report.with_columns(
    (pl.col('saldo_total') / pl.col('tasa_factura')).round(2)
    .alias('saldo_dolar')
)
financing_report = financing_report.drop(['fecha_right', 'saldo_total', 'tasa_factura'])
financing_report = financing_report.with_columns(
    pl.when(pl.col('saldo_dolar').is_null())
    .then(0)
    .otherwise(1)
    .alias('factura_pagada')
)
financing_report_backup = financing_report
################################################################################################################
# Trabajando con la data para los telefonos vendidos
################################################################################################################
phones_report = telefonos_vendidos_cashea
################################################################################################################
# Trabajando con la data para los paquetes enviados por MRW
################################################################################################################
mrw_packs_report = envios_mrw

In [ ]:
# Cambio de nombres y filtros
bill_report = bill_report_backup.rename({
    'emission_date': 'fecha',
    'register_hour': 'hora_registro',
    'document_no': 'factura',
    'amount': 'cantidad',
    'description': 'producto',
    'description_right': 'vendedor',
    'dollar_net': 'monto'
})
bill_report = bill_report.filter(
    ~(
    pl.col('producto').str.to_lowercase().str.starts_with('combo') |
    pl.col('producto').str.to_lowercase().str.starts_with('tarifa delivery')
    )
)
financing_report = financing_report_backup.rename({
    'monto_dolar': 'financiamiento',
    'saldo_dolar': 'monto_pagado'
})

In [ ]:
# Verificaciones
#print(f'Numero total de filas en las facturaciones de cashea {len(envios_mrw)}')
#print(f'Facturaciones unicas de cashea {envios_mrw.unique().height}')

In [ ]:
# Guardado de archivo
bill_report.write_csv(
    r'C:\Users\POWERBI\Desktop\Entornos\cashea_stats\reports\cashea_bills.csv',
    separator=',',
    include_header=True,
    decimal_comma=False,
    date_format='%Y-%m-%d',
    null_value=''
)
financing_report.write_csv(
    r'C:\Users\POWERBI\Desktop\Entornos\cashea_stats\reports\cashea_financing.csv',
    separator=',',
    include_header=True,
    decimal_comma=False,
    date_format='%Y-%m-%d',
    null_value=''
)
phones_report.write_csv(
    r'C:\Users\POWERBI\Desktop\Entornos\cashea_stats\reports\cashea_phones.csv',
    separator=',',
    include_header=True,
    decimal_comma=False,
    date_format='%Y-%m-%d',
    null_value=''
)
mrw_packs_report.write_csv(
    r'C:\Users\POWERBI\Desktop\Entornos\cashea_stats\reports\cashea_mrw.csv',
    separator=',',
    include_header=True,
    decimal_comma=False,
    date_format='%Y-%m-%d',
    null_value=''
)
print('Libros guardados')